# Обучение Detail Refiner (Этап 2)

Refiner поверх base autoencoder: уточняет детали через uncertainty-gated residual.

**Предусловие:** Запустите train_colab.ipynb первым — нужен best_autoencoder.pt на Drive.

In [ ]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код
%cd /content
!rm -rf MultiView_3D_Prediction
!git clone https://github.com/3x6dll9ff/MultiView_3D_Prediction.git
%cd MultiView_3D_Prediction

In [ ]:
# 3. Зависимости
!pip install -r requirements.txt

In [ ]:
# 4. Подготовка: данные + базовая модель с Drive
import os, shutil

# Данные
!rm -rf /content/data
!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/data

# Базовая модель
DRIVE_DIR = '/content/drive/MyDrive/MultiView_Results'
base_model_drive = os.path.join(DRIVE_DIR, 'best_autoencoder.pt')
base_model_local = 'results/best_autoencoder.pt'

os.makedirs('results', exist_ok=True)
if os.path.exists(base_model_drive):
    shutil.copy2(base_model_drive, base_model_local)
    print(f"Base model скопирован: {base_model_local}")
else:
    print("ОШИБКА: best_autoencoder.pt не найден на Drive!")
    print("Запустите train_colab.ipynb первым.")

In [ ]:
# 5. Обучение Refiner
!DATA_PATH=$(find /content/data -name "metadata.csv" | head -n 1 | xargs dirname) && \
 echo "Используем данные из: $DATA_PATH" && \
 python3 src/train_refiner.py \
    --data_dir "$DATA_PATH" \
    --base_model results/best_autoencoder.pt \
    --input_mode tri \
    --epochs 50 \
    --batch_size 8 \
    --lr 3e-4 \
    --num_workers 2

In [ ]:
# 6. Сохраняем результаты на Drive
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/MultiView_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

for f in ['best_refiner.pt']:
    src = os.path.join('results', f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, f))
        print(f"Сохранено: {f}")

metrics_dir = os.path.join(DRIVE_DIR, 'metrics')
os.makedirs(metrics_dir, exist_ok=True)
for f in os.listdir('results/metrics'):
    src = os.path.join('results/metrics', f)
    if 'refiner' in f and os.path.isfile(src):
        shutil.copy2(src, os.path.join(metrics_dir, f))
        print(f"Сохранено: metrics/{f}")

print(f"\nРефайнер на Drive: {DRIVE_DIR}/best_refiner.pt")